## 0) DOE 데이터 재생성 (선택)

새 DOE 데이터를 만들 때만 실행합니다.

- 실행 환경: **호스트 Motor-CAD** (Docker 아님)
- 생성 결과:
  - `case_NNNN/*.mot` : 케이스별 기준 모델 복사본
  - `case_NNNN/FEResultsData/` : Motor-CAD 원본 결과 루트
  - `case_NNNN/doe_condition.json` : DOE 변수/적용값
  - `case_NNNN/postproc/*.txt` : 메인 DOE 배치 산출물
  - `case_NNNN/postproc/*.h5` : 필요 시 0-D 후속 셀에서 생성

권장 순서:
1. 0-B 설정 셀에서 경로/샘플 수/단계를 지정
2. `RUN_DOE=True`로 변경
3. 0-C 실행 셀 실행 (`solve + txt`)
4. 학습까지 이어갈 때만 `RUN_H5_EXPORT=True`로 바꿔 0-D 실행

**중간 중단 후 복구:**
- 0-E-1 셀로 상태 점검 (`.mes` 있음 / `.txt` 없음 케이스 식별)
- 0-E-2 셀에서 `RUN_REPAIR=True`로 바꿔 `.txt` 재내보내기

In [ ]:

# 0-B) DOE 설정 셀 (Host Motor-CAD)
import json
import os
import sys
import importlib
from pathlib import Path

import ansys.motorcad.core as pymotorcad

ROOT = Path.cwd()
EMACH_ROOT = ROOT / "eMach"
if str(EMACH_ROOT) not in sys.path:
    sys.path.insert(0, str(EMACH_ROOT))

from tools.pyutils.sweep import DOEAxis, DOEPoint, build_doe_lhs
import tools.motorCAD.pyMCAD.doe_batch as _doe_batch
importlib.reload(_doe_batch)
from tools.motorCAD.pyMCAD.doe_batch import doe_batch_run, doe_h5_batch_from_txt

# ----- 사용자 설정 -----
BASE_MOT = r"D:\KDH\Sim_4SolverX\TestCAD1.mot"
DOE_OUT = Path(r"D:\KDH\Sim_4SolverX\DOE4TrainingData")
DOE_OUT.mkdir(parents=True, exist_ok=True)

N_SAMPLES = 40
LHS_SEED = 42
LHS_CRITERION = "maximin"

PHASES = ["solve", "export_txt"]
# backward compatibility가 필요하면 아래 legacy 경로 사용:
# PHASES = ["solve", "export"]

FIRST_STEP = 1
FINAL_STEP = 45
MAG_H5_MESH_COORDS = "by_step_moving_nodes"
MAG_COLUMNS = "RegCode,Bx,By,A,J,Je"
PLOT_MODE = "none"

# 병렬 실행 워커 수 (1=직렬, 2 이상=병렬)
# Windows + Motor-CAD COM 환경에서는 2부터 시작하는 것을 권장
PARALLEL_WORKERS = 2

# 안전 스위치
RUN_DOE = True
RUN_H5_EXPORT = False

print(f"ROOT={ROOT}")
print(f"EMACH_ROOT={EMACH_ROOT}")
print(f"BASE_MOT={BASE_MOT}")
print(f"DOE_OUT={DOE_OUT}")
print(f"PHASES={PHASES}, N_SAMPLES={N_SAMPLES}, RUN_DOE={RUN_DOE}")
print(f"RUN_H5_EXPORT={RUN_H5_EXPORT}")
print(f"PARALLEL_WORKERS={PARALLEL_WORKERS}")


ROOT=d:\KDH\NvidiaNemo
EMACH_ROOT=d:\KDH\NvidiaNemo\eMach
BASE_MOT=D:\KDH\Sim_4SolverX\TestCAD1.mot
DOE_OUT=D:\KDH\Sim_4SolverX\DOE4TrainingData
PHASES=['solve', 'export_txt'], N_SAMPLES=40, RUN_DOE=True
RUN_H5_EXPORT=False
PARALLEL_WORKERS=8


In [3]:

# 0-C) 실행 셀: DOE 데이터 생성 (solve + txt-only)
#
# [첫 실행]
#   → Motor-CAD 접속 → build_doe_lhs → doe_grid.json 저장 → 전체 케이스 실행
#
# [재실행 — 0-E-1 통합]
#   → doe_grid.json 로드 (build_doe_lhs 생략)
#   → doe_scan_status 로 상태 점검
#   → needs_solve / not-started 케이스만 doe_batch_run
#   → needs_txt 케이스는 doe_repair_missing_txt 로 txt 재내보내기 (병렬 지원)
#   → 모두 complete 이면 신규 실행 없이 요약만 출력

# ── helpers ──────────────────────────────────────────────────────────────────
def _get_var(mc, name):
    """Motor-CAD get_variable 반환값 언래핑.
    enable_success_variable=True 환경에서는 (returncode, value) 튜플을 반환하므로
    value만 추출합니다."""
    v = mc.get_variable(name)
    return v[1] if isinstance(v, tuple) and len(v) == 2 and isinstance(v[0], int) else v


def _save_doe_grid(grid, path, meta=None):
    """DOEPoint 리스트를 JSON으로 직렬화해 저장합니다."""
    data = {
        "meta": meta or {},
        "points": [
            {"geometry": pt.geometry, "electrical": pt.electrical, "index": pt.index}
            for pt in grid
        ],
    }
    Path(path).write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")


def _load_doe_grid(path):
    """JSON에서 DOEPoint 리스트를 복원합니다."""
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    return [
        DOEPoint(geometry=p["geometry"], electrical=p["electrical"], index=p["index"])
        for p in data["points"]
    ]

# ── 실행 본체 ────────────────────────────────────────────────────────────────
DOE_GRID_JSON = DOE_OUT / "doe_grid.json"

if not RUN_DOE:
    print("RUN_DOE=False 입니다. 설정 확인 후 True로 바꿔 실행하세요.")
else:
    importlib.reload(_doe_batch)
    from tools.motorCAD.pyMCAD.doe_batch import (
        doe_batch_run, doe_scan_status, doe_repair_missing_txt,
    )

    # ── Step 1: DOE 그리드 로드 / 신규 생성 ─────────────────────────────
    if DOE_GRID_JSON.exists():
        doe_grid = _load_doe_grid(DOE_GRID_JSON)
        print(f"[0-C] 기존 DOE 그리드 로드: {len(doe_grid)}개 포인트 ({DOE_GRID_JSON.name})")
        print("      → build_doe_lhs 생략 (Motor-CAD 접속 지연)")
    else:
        # 기준값 읽기용 Motor-CAD 접속
        try:
            mc = pymotorcad.MotorCAD(open_new_instance=False)
            print("Motor-CAD: existing instance connected")
        except Exception as _e0:
            print(f"[warn] existing instance connect failed: {_e0}")
            print("Motor-CAD: launching new instance...")
            mc = pymotorcad.MotorCAD(open_new_instance=True)

        mc.load_from_file(BASE_MOT)
        rb_base = float(_get_var(mc, "Ratio_Bore"))
        rsd_base = float(_get_var(mc, "Ratio_SlotDepth_ParallelSlot"))

        all_axes = [
            DOEAxis("Ratio_Bore", round(rb_base * 0.90, 4), round(rb_base * 1.10, 4), steps=0),
            DOEAxis("Ratio_SlotDepth_ParallelSlot", round(rsd_base * 0.85, 4), round(rsd_base * 1.15, 4), steps=0),
            DOEAxis("PeakCurrent", 10.0, 650.53, steps=0),
            DOEAxis("PhaseAdvance", 0.0, 90.0, steps=0),
        ]
        doe_grid = build_doe_lhs(
            axes=all_axes, n_samples=N_SAMPLES, seed=LHS_SEED, criterion=LHS_CRITERION,
        )
        _save_doe_grid(doe_grid, DOE_GRID_JSON, meta={
            "n_samples": N_SAMPLES, "seed": LHS_SEED, "criterion": LHS_CRITERION,
            "axes": [{"name": ax.name, "min": ax.min_val, "max": ax.max_val} for ax in all_axes],
        })
        print(f"[0-C] 신규 DOE 그리드 생성: {len(doe_grid)}개 포인트 → {DOE_GRID_JSON.name}")

    # ── Step 2: 상태 점검 (0-E-1 통합) ──────────────────────────────────
    scan = doe_scan_status(DOE_OUT, verbose=True)
    _scanned_idx = {c["index"] for c in scan["cases"]}
    _not_started = [pt.index for pt in doe_grid if pt.index not in _scanned_idx]
    _solve_idx = _not_started + scan["solve_needed"]  # 미시작 + 해석 미완료
    _repair_idx = scan["repair_needed"]  # mes 있음, txt 없음

    print(json.dumps({
        "summary": scan["summary"],
        "not_started": len(_not_started),
        "solve_needed": len(_solve_idx),
        "repair_needed": len(_repair_idx),
    }, indent=2, ensure_ascii=False))

    if not _solve_idx and not _repair_idx:
        # 모든 케이스 완료 — 신규 실행 없음
        print("✓ 모든 케이스가 완료 상태입니다. 신규 실행 없음.")
        _txt_ready = scan["summary"].get("complete", 0)
        manifest = {}
    else:
        # Motor-CAD 인스턴스 확보 (아직 없으면 접속)
        if "mc" not in globals():
            try:
                mc = pymotorcad.MotorCAD(open_new_instance=False)
                print("Motor-CAD: existing instance connected")
            except Exception as _e1:
                print(f"[warn] {_e1}")
                mc = pymotorcad.MotorCAD(open_new_instance=True)
                print("Motor-CAD: launching new instance...")
            mc.load_from_file(BASE_MOT)

        # ── Step 3-a: 미시작 / needs_solve 케이스 해석 ───────────────
        if _solve_idx:
            _solve_set = set(_solve_idx)
            _solve_grid = [pt for pt in doe_grid if pt.index in _solve_set]
            print(f"[0-C] solve → {len(_solve_grid)}개 케이스 실행...")
            manifest = doe_batch_run(
                mc, _solve_grid,
                base_mot=BASE_MOT, doe_out_root=DOE_OUT, phases=PHASES,
                first_step=FIRST_STEP, final_step=FINAL_STEP,
                mag_columns=MAG_COLUMNS, mag_h5_mesh_coords=MAG_H5_MESH_COORDS,
                plot_mode=PLOT_MODE, parallel_workers=PARALLEL_WORKERS,
            )
        else:
            manifest = {}

        # ── Step 3-b: needs_txt 케이스 txt 재내보내기 ────────────────
        if _repair_idx:
            print(
                f"[0-C] needs_txt 복구 → {len(_repair_idx)}개 케이스 "
                f"(parallel_workers={PARALLEL_WORKERS})..."
            )
            _repair = doe_repair_missing_txt(
                mc, DOE_OUT, case_indices=_repair_idx,
                first_step=FIRST_STEP, final_step=FINAL_STEP,
                mag_columns=MAG_COLUMNS, mag_h5_mesh_coords=MAG_H5_MESH_COORDS,
                plot_mode=PLOT_MODE, parallel_workers=PARALLEL_WORKERS,
                verbose=True,
            )
            print(json.dumps({
                "repair_repaired": _repair["repaired"],
                "repair_failed": _repair["failed"],
            }, indent=2, ensure_ascii=False))

        # 최종 완료 수 재집계
        _final_scan = doe_scan_status(DOE_OUT, verbose=False)
        _txt_ready = _final_scan["summary"].get("complete", 0)

    print(json.dumps({
        "base_mot": BASE_MOT,
        "doe_out": str(DOE_OUT),
        "doe_grid_json": str(DOE_GRID_JSON),
        "n_cases": len(doe_grid),
        "txt_ready_cases": _txt_ready,
        "failed": len(manifest.get("failed", [])),
    }, indent=2, ensure_ascii=False))


Motor-CAD: existing instance connected
[0-C] 신규 DOE 그리드 생성: 40개 포인트 → doe_grid.json

===== DOE 상태 점검: D:\KDH\Sim_4SolverX\DOE4TrainingData =====
  전체 케이스: 0
{
  "summary": {
    "complete": 0,
    "needs_txt": 0,
    "needs_solve": 0,
    "empty": 0
  },
  "not_started": 40,
  "solve_needed": 40,
  "repair_needed": 0
}
[0-C] solve → 40개 케이스 실행...
[doe_batch] parallel_workers=8
  [ERROR] case_0000: A process in the process pool was terminated abruptly while the future was running or pending.
  [ERROR] case_0001: A process in the process pool was terminated abruptly while the future was running or pending.
  [ERROR] case_0002: A process in the process pool was terminated abruptly while the future was running or pending.
  [ERROR] case_0003: A process in the process pool was terminated abruptly while the future was running or pending.
  [ERROR] case_0004: A process in the process pool was terminated abruptly while the future was running or pending.
  [ERROR] case_0005: A process in the pr

### 0-D) 선택 셀: TXT -> H5 변환

학습/추론 섹션으로 바로 이어갈 때만 실행합니다.

- 실행 환경: 호스트 Python만 필요
- 입력: 0-C에서 생성된 `case_NNNN/postproc/Mag_*.txt`
- 출력: `case_NNNN/postproc/Mag_*.h5` + `doe_manifest.json`의 `h5_paths` 갱신

In [ ]:
if not RUN_H5_EXPORT:
    print("RUN_H5_EXPORT=False 입니다. H5가 필요할 때만 True로 바꿔 실행하세요.")
else:
    h5_summary = doe_h5_batch_from_txt(
        DOE_OUT,
        mag_h5_mesh_coords=MAG_H5_MESH_COORDS,
        verbose=True,
    )

    h5_ready = sum(
        1 for case in h5_summary.get("cases", [])
        if not case.get("error")
    )

    print(json.dumps({
        "doe_out": str(DOE_OUT),
        "requested_cases": len(h5_summary.get("cases", [])),
        "h5_ready_cases": h5_ready,
    }, indent=2, ensure_ascii=False))


### 0-E) 수동 복구: DOE 상태 재점검 + TXT 재내보내기

0-C 재실행만으로 복구되지 않는 경우(예: `mc` 접속 없이 상태만 확인하고 싶을 때)에 사용합니다.

> **일반 복구 흐름:** 0-C 셀 재실행만으로 충분합니다 (`doe_grid.json` 이 있으면 자동 상태 점검 → 미완성 케이스만 실행).

**0-E 점검 항목:**
| 상태 | 의미 | 자동 조치 (0-C 재실행 시) |
|------|------|--------------------------|
| `complete` | `.mes` + `.txt` 모두 있음 | 없음 |
| `needs_txt` | `.mes` 있음, `.txt` 없음 | `doe_repair_missing_txt` 자동 호출 |
| `needs_solve` | `.mes` 없음 (해석 미완료) | `doe_batch_run` 자동 호출 |
| `empty` | 초기화 흔적 없음 | 무시 |

**0-E 수동 실행 순서 (0-C 재실행이 어려울 때):**
1. `0-E-1` 상태 점검 셀 실행 → 복구 대상 인덱스 확인
2. `RUN_REPAIR=True`로 변경 후 `0-E-2` 복구 셀 실행


In [ ]:
# 0-E-1) DOE 상태 점검
# 이 셀은 Motor-CAD 없이 실행 가능합니다.
import json
import importlib
from pathlib import Path

# 0-B 셀을 먼저 실행하지 않은 경우를 위한 독립 경로 설정
_doe_out_check = globals().get("DOE_OUT", Path(r"D:\KDH\Sim_4SolverX\DOE4TrainingData"))

import tools.motorCAD.pyMCAD.doe_batch as _doe_batch
importlib.reload(_doe_batch)
from tools.motorCAD.pyMCAD.doe_batch import doe_scan_status

scan_result = doe_scan_status(_doe_out_check, verbose=True)

# 결과 요약 출력
print(json.dumps({
    "doe_out": str(_doe_out_check),
    "summary": scan_result["summary"],
    "repair_needed_count": len(scan_result["repair_needed"]),
    "solve_needed_count": len(scan_result["solve_needed"]),
    "repair_needed_indices": scan_result["repair_needed"][:20],   # 최대 20개만 표시
    "solve_needed_indices": scan_result["solve_needed"][:20],
}, indent=2, ensure_ascii=False))


In [ ]:

# 0-E-2) TXT 재내보내기 복구 셀 (mes 있음 + txt 없는 케이스)
# 안전 스위치 — True로 변경해야 실행됩니다.
RUN_REPAIR = True

if not RUN_REPAIR:
    print("RUN_REPAIR=False 입니다. 복구할 케이스가 있으면 True로 바꿔 실행하세요.")
    print(f"  복구 대상 케이스: {scan_result.get('repair_needed', [])}")
else:
    import importlib
    import tools.motorCAD.pyMCAD.doe_batch as _doe_batch_repair
    importlib.reload(_doe_batch_repair)
    from tools.motorCAD.pyMCAD.doe_batch import doe_repair_missing_txt

    # Motor-CAD 인스턴스 준비 (0-C에서 mc가 이미 있으면 재사용)
    if "mc" not in globals():
        import ansys.motorcad.core as pymotorcad
        try:
            mc = pymotorcad.MotorCAD(open_new_instance=False)
            print("Motor-CAD: existing instance connected")
        except Exception as _e:
            print(f"[warn] {_e}")
            mc = pymotorcad.MotorCAD(open_new_instance=True)
            print("Motor-CAD: new instance launched")

    # 복구 대상 인덱스 (None = 점검에서 발견된 needs_txt 전체)
    REPAIR_CASE_INDICES = scan_result.get("repair_needed") or None
    REPAIR_PARALLEL_WORKERS = int(globals().get("PARALLEL_WORKERS", 1))

    repair_summary = doe_repair_missing_txt(
        mc,
        _doe_out_check,
        case_indices=REPAIR_CASE_INDICES,
        first_step=globals().get("FIRST_STEP", 1),
        final_step=globals().get("FINAL_STEP", 45),
        mag_columns=globals().get("MAG_COLUMNS", "RegCode,Bx,By,A,J,Je"),
        mag_h5_mesh_coords=globals().get("MAG_H5_MESH_COORDS", "by_step_moving_nodes"),
        plot_mode="none",
        parallel_workers=REPAIR_PARALLEL_WORKERS,
        dry_run=False,
        verbose=True,
    )

    print(json.dumps({
        "doe_out": str(_doe_out_check),
        "parallel_workers": REPAIR_PARALLEL_WORKERS,
        "repaired": repair_summary["repaired"],
        "failed": repair_summary["failed"],
        "skipped": repair_summary["skipped"],
    }, indent=2, ensure_ascii=False))


# Phase 1 Tutorial — SymMGN PBC 전체 검증

이 노트북은 Phase 1 (Static SymMGN with PBC) 완료 증거를 생성합니다.

**핵심 원칙:**
- 로컬 커널은 오케스트레이션/시각화만 수행
- 모델 학습/추론은 `motor_compare` 컨테이너 내부에서 실행

**실행 순서:**
| Part | 내용 | 셀 |
|------|------|----|
| 0 | (선택) DOE 데이터 재생성 + H5 후속 변환 | 0-B ~ 0-D |
| A | 환경 + Docker + GPU | 1-4 |
| B | Contract / PBC / Overfit 검증 | 5-7 |
| C | **PBC 경계 가시화** (학습 전 확인) | 8 |
| D | 3-case DOE smoke test | 9-10 |
| E | **Full 40-case 학습** | 11 |
| F | **Full 40-case 추론 + 시각화** | 12-13 |
| G | Phase 1 완료 Evidence | 14

## 1) 환경 설정 및 라이브러리 임포트

로컬에서는 실행 제어/결과 파싱만 수행합니다.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd()
CONTAINER_NAME = os.environ.get("PHYSICSNEMO_CONTAINER", "motor_compare")
LOG_DIR = ROOT / "logs" / "tutorial"
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT={ROOT}")
print(f"CONTAINER_NAME={CONTAINER_NAME}")
print(f"LOG_DIR={LOG_DIR}")

## 2) Docker 실행 헬퍼

학습/추론/테스트는 모두 컨테이너 내부에서 실행합니다.

In [ ]:
def run_local(cmd: str, check: bool = True) -> subprocess.CompletedProcess:
    print(f"[local] {cmd}")
    return subprocess.run(cmd, shell=True, text=True, capture_output=True, check=check)


def run_docker(cmd: str, check: bool = True) -> subprocess.CompletedProcess:
    # Use double-quoted bash -lc payload to avoid breaking on single quotes in Python code.
    payload = f"cd /workspace/app && {cmd}"
    payload = payload.replace("\\", "\\\\").replace('"', '\\"')
    full_cmd = f'docker exec {CONTAINER_NAME} bash -lc "{payload}"'
    print(f"[docker] {cmd}")
    return run_local(full_cmd, check=check)


def save_log(name: str, cp: subprocess.CompletedProcess) -> Path:
    path = LOG_DIR / name
    text = []
    text.append(f"$ returncode={cp.returncode}\n")
    if cp.stdout:
        text.append("\n[stdout]\n")
        text.append(cp.stdout)
    if cp.stderr:
        text.append("\n[stderr]\n")
        text.append(cp.stderr)
    path.write_text("".join(text), encoding="utf-8")
    print(f"saved: {path}")
    return path

## 3) GPU-enabled Docker Compose 기동

GPU, IPC, ulimit 설정이 적용되도록 컨테이너를 재생성합니다.

In [ ]:
cp_compose_up = run_local("docker compose up -d --force-recreate", check=False)
save_log("00_compose_up.log", cp_compose_up)
print(cp_compose_up.stdout)
if cp_compose_up.stderr.strip():
    print(cp_compose_up.stderr)
if cp_compose_up.returncode != 0:
    raise RuntimeError("GPU-enabled docker compose up 에 실패했습니다. compose 설정과 Docker GPU 런타임을 확인하세요.")

## 4) GPU / PyTorch 확인 (컨테이너 내부)

In [ ]:
cp_nvidia_smi = run_docker("nvidia-smi", check=False)
save_log("01_nvidia_smi.log", cp_nvidia_smi)
print(cp_nvidia_smi.stdout or cp_nvidia_smi.stderr)

cp_torch = run_docker(
    "python -c \"import torch; print('torch', torch.__version__); print('cuda', torch.cuda.is_available()); print('device_count', torch.cuda.device_count()); print('cuda_version', torch.version.cuda)\"",
    check=False,
 )
save_log("01_torch_env.log", cp_torch)
print(cp_torch.stdout)
if cp_torch.stderr.strip():
    print(cp_torch.stderr)
if cp_nvidia_smi.returncode != 0 or cp_torch.returncode != 0 or "cuda False" in cp_torch.stdout:
    raise RuntimeError(
        "GPU가 컨테이너에 노출되지 않았습니다. 01_nvidia_smi.log 와 01_torch_env.log를 확인하세요."
    )

## 5) Contract / PBC / Overfit 검증 게이트

Phase 1 계약 경계 테스트 → PBC 경계/계약 호스트 테스트 → Overfit-Single 게이트 → PBC bundle 테스트

In [ ]:
cp_contract = run_docker(
    "PYTHONPATH=/workspace/app pytest -q tests/test_phase1_contract_boundaries.py",
    check=False,
 )
save_log("02_contract_tests.log", cp_contract)
print(cp_contract.stdout)
if cp_contract.stderr.strip():
    print(cp_contract.stderr)
if cp_contract.returncode != 0:
    raise RuntimeError("Contract 경계 테스트 실패. 로그(02_contract_tests.log)를 확인하세요.")

In [ ]:
import subprocess

# torch 없이 돌아가는 경계/계약 테스트는 호스트 venv에서 허용
# (phase1_static.pbc_boundary, pbc_contracts 는 scipy만 필요)
py_exec = r"c:/Users/moa/.ansys_python_venvs/PyMotorEnv_310/Scripts/python.exe"
cp_pbc_host_tests = subprocess.run(
    [
        py_exec, "-m", "pytest", "-q",
        "tests/test_phase1_pbc_boundary.py",
        "tests/test_phase1_pbc_contracts.py",
    ],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(cp_pbc_host_tests.stdout)
if cp_pbc_host_tests.returncode != 0:
    print("[stderr]", cp_pbc_host_tests.stderr[:600])
print("exit_code:", cp_pbc_host_tests.returncode)

In [ ]:
import subprocess, json, textwrap

OVERFIT_SCRIPT = textwrap.dedent("""
import sys, json, tempfile, subprocess
import numpy as np
from pathlib import Path

try:
    angles = np.deg2rad(np.array([0.0, 22.5, 45.0], dtype=np.float64))
    inner = np.stack([np.cos(angles), np.sin(angles)], axis=1).astype(np.float32)
    pos = inner[None]
    node_type = np.ones((1, 3, 1), dtype=np.float32)
    edges = np.array([[0,1],[1,2],[2,0],[1,0],[2,1],[0,2]], dtype=np.int64)
    ie = edges.T[None]
    pe = np.zeros((1, 2, 0), dtype=np.int64)
    pa = np.zeros((1, 0, 1), dtype=np.float32)
    y = np.random.RandomState(42).randn(1, 3, 4).astype(np.float32)

    with tempfile.TemporaryDirectory() as td:
        npz_path = Path(td) / "overfit.npz"
        np.savez(npz_path, pos=pos, node_type_onehot=node_type,
                 interior_edge_index=ie, pbc_edge_index=pe,
                 pbc_edge_attr=pa, y=y)

        r = subprocess.run(
            [sys.executable, "-m", "phase1_static.train",
             "--input-format", "npz", "--data", str(npz_path),
             "--overfit-single", "--epochs", "150",
             "--lr", "1e-2", "--hidden-dim", "32",
             "--seed", "42", "--overfit-target", "1e-2"],
            capture_output=True, text=True, cwd="/workspace/app",
        )
        # train.py logs to stderr via logging module
        combined = r.stdout + r.stderr
        epoch_lines = [l for l in combined.splitlines() if "epoch=" in l]
        overfit_lines = [l for l in combined.splitlines() if "Overfit" in l or "overfit" in l]
        print(json.dumps({
            "returncode": r.returncode,
            "last_epoch": epoch_lines[-1] if epoch_lines else "",
            "overfit_gate": overfit_lines[-1] if overfit_lines else "",
        }))
except Exception as exc:
    import traceback
    print(json.dumps({"error": str(exc), "tb": traceback.format_exc()[-400:]}))
""")

cp_overfit = subprocess.run(
    ["docker", "exec", CONTAINER_NAME, "python", "-c", OVERFIT_SCRIPT],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)

print("=== Overfit-Single (Docker) ===")
for line in cp_overfit.stdout.splitlines():
    try:
        d = json.loads(line)
        print(json.dumps(d, indent=2, ensure_ascii=False))
    except Exception:
        print(line)
print("exit_code:", cp_overfit.returncode)

In [ ]:
import subprocess

# pbc_bundle 보강 테스트 + 기존 PBC 테스트 전체 확인
py_exec = r"c:/Users/moa/.ansys_python_venvs/PyMotorEnv_310/Scripts/python.exe"
cp_all_pbc = subprocess.run(
    [
        py_exec, "-m", "pytest", "-q",
        "tests/test_phase1_pbc_bundle.py",
        "tests/test_phase1_pbc_boundary.py",
        "tests/test_phase1_pbc_contracts.py",
        "tests/test_phase1_pbc_pairing.py",
        "tests/test_phase1_pbc_prior.py",
    ],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(cp_all_pbc.stdout)
if cp_all_pbc.returncode != 0:
    print("[stderr]", cp_all_pbc.stderr[:600])
print("exit_code:", cp_all_pbc.returncode)

## 8) PBC 경계 가시화 — 학습 전 확인

실제 DOE 메쉬에서 추출한 master/slave 경계선과 PBC 엣지를 시각화합니다.
학습 전에 PBC topology가 올바르게 구성되었는지 **반드시** 눈으로 확인합니다.

표시 내용:
- 전체 노드 분포 (1/8 섹터)
- master / slave boundary line
- 실제 pbc_edge overlay (anti-periodic, edge_attr=-1.0)
- case별 match ratio, pair 수, group labels

In [ ]:
import importlib
import json

from IPython.display import display

pbc_boundary_module = importlib.import_module(
    "phase1_static.pbc_boundary"
 )
pbc_candidate_module = importlib.import_module(
    "postproc_interop.model.PBCBoundaryCandidate"
 )
pbc_case_module = importlib.import_module(
    "postproc_interop.model.PBCVisualizationCase"
 )
pbc_module = importlib.import_module("postproc_interop.pbc")
pbc_bridge_module = importlib.import_module(
    "postproc_interop.bridges.MotorCADPBCVisualizationBridge"
 )

for module in (
    pbc_boundary_module,
    pbc_candidate_module,
    pbc_case_module,
    pbc_module,
    pbc_bridge_module,
 ):
    importlib.reload(module)

MotorCADPBCVisualizationBridge = pbc_bridge_module.MotorCADPBCVisualizationBridge

PBC_VIS_CASE_INDICES = list(globals().get("DOE_CASE_INDICES", [0, 1])[:2])
if len(PBC_VIS_CASE_INDICES) < 2:
    PBC_VIS_CASE_INDICES = [0, 1]

PBC_VIS_SOURCE_TYPES = list(globals().get("DOE_SOURCE_FILE_TYPES", ["OnLoadTorque"]))
PBC_VIS_OUT_DIR = ROOT / "results" / "pbc_case_vis"

pbc_bridge = MotorCADPBCVisualizationBridge(ROOT)
pbc_cases, skipped_cases = pbc_bridge.collect_cases(
    case_indices=PBC_VIS_CASE_INDICES,
    source_file_types=PBC_VIS_SOURCE_TYPES,
    data_dir=ROOT / "doe_data",
    output_dir=PBC_VIS_OUT_DIR,
)

summary = {
    "case_indices": [int(case.case_idx) for case in pbc_cases],
    "source_file_types": sorted(
        {case.source_file_type for case in pbc_cases}
    ),
    "pairs": {
        str(case.case_idx): int(case.pbc_forward_index.shape[1])
        for case in pbc_cases
    },
    "groups": {
        str(case.case_idx): list(case.group_labels)
        for case in pbc_cases
    },
    "skipped_cases": skipped_cases,
}

print(json.dumps(summary, indent=2, ensure_ascii=False))
display(
    pbc_bridge.build_case_selector_widget(
        cases=pbc_cases,
        default_group="all",
        default_show_nodes=False,
        default_show_pbc_edges=False,
    )
)

## 9) SymMGN 3-case DOE Smoke Test

Full training 전에 3개 case로 파이프라인이 올바르게 작동하는지 확인합니다.

| 항목 | 값 |
|------|----|
| 모델 | SymMGN (anti-periodic PBC) |
| Case | 3개 (DOE index 0, 1, 2) |
| Source file type | OnLoadTorque |
| Max steps/case | 2 |
| Epochs | 2 |

In [ ]:
import json

DOE_CASE_INDICES = [0, 1, 2]
DOE_SOURCE_FILE_TYPES = ["OnLoadTorque"]
DOE_MAX_STEPS_PER_CASE = 2
DOE_EPOCHS = 2
DOE_BATCH_SIZE = 2
DOE_HIDDEN_DIM = 64
DOE_SEED = 42

SYM_CKPT_PATH = ROOT / "results" / "symm_mgn_doe_onloadtorque.pt"
SYM_TRAIN_LOG = "11_symm_train_doe.log"

train_cmd = " ".join(
    [
        "python -m phase1_static.train",
        "--input-format doe",
        "--data-dir doe_data",
        "--case-indices " + " ".join(str(idx) for idx in DOE_CASE_INDICES),
        "--source-file-types " + " ".join(DOE_SOURCE_FILE_TYPES),
        f"--max-steps-per-case {DOE_MAX_STEPS_PER_CASE}",
        f"--epochs {DOE_EPOCHS}",
        f"--batch-size {DOE_BATCH_SIZE}",
        f"--hidden-dim {DOE_HIDDEN_DIM}",
        f"--seed {DOE_SEED}",
        f"--ckpt-out results/{SYM_CKPT_PATH.name}",
    ]
)

cp_symm_train = run_docker(train_cmd, check=False)
save_log(SYM_TRAIN_LOG, cp_symm_train)

combined_train = (cp_symm_train.stdout or "") + "\n" + (cp_symm_train.stderr or "")
epoch_lines = [line for line in combined_train.splitlines() if "epoch=" in line]
pbc_skip_lines = [line for line in combined_train.splitlines() if "PBC boundary match skipped" in line]

print("=== SymMGN DOE 학습 (Docker) ===")
print(
    json.dumps(
        {
            "returncode": cp_symm_train.returncode,
            "case_indices": DOE_CASE_INDICES,
            "source_file_types": DOE_SOURCE_FILE_TYPES,
            "last_epoch": epoch_lines[-1] if epoch_lines else "",
            "pbc_skip_count": len(pbc_skip_lines),
            "ckpt_saved": SYM_CKPT_PATH.exists(),
            "ckpt_size_kb": round(SYM_CKPT_PATH.stat().st_size / 1024, 1) if SYM_CKPT_PATH.exists() else 0,
            "log_file": str(LOG_DIR / SYM_TRAIN_LOG),
        },
        indent=2,
        ensure_ascii=False,
    )
)

if cp_symm_train.returncode != 0:
    print("=== train stderr tail ===")
    print("\n".join(combined_train.splitlines()[-40:]))
    raise RuntimeError("DOE 실데이터 SymMGN 학습 실패. 11_symm_train_doe.log를 확인하세요.")

## 10) Smoke Test 추론 + 시각화

3-case smoke 모델로 case 0의 첫 step을 추론하고 GT vs Pred 시각화를 확인합니다.

In [ ]:
import json

DOE_SOURCE_FILE_TYPES = globals().get("DOE_SOURCE_FILE_TYPES", ["OnLoadTorque"])
SYM_CKPT_PATH = globals().get("SYM_CKPT_PATH", ROOT / "results" / "symm_mgn_doe_onloadtorque.pt")
DOE_INFER_CASE_IDX = globals().get("DOE_CASE_INDICES", [0])[0]
DOE_INFER_MAX_STEPS = 1
SYM_INFER_NPZ_PATH = ROOT / "results" / f"symm_mgn_case{DOE_INFER_CASE_IDX:04d}_onloadtorque_step1.npz"
SYM_INFER_LOG = "12_symm_infer_doe.log"

infer_cmd = " ".join(
    [
        "python infer_phase1_pbc.py",
        f"--ckpt /workspace/app/results/{SYM_CKPT_PATH.name}",
        "--data-dir /workspace/app/doe_data",
        f"--case-idx {DOE_INFER_CASE_IDX}",
        "--source-file-types " + " ".join(DOE_SOURCE_FILE_TYPES),
        f"--max-steps {DOE_INFER_MAX_STEPS}",
        "--batch-size 1",
        f"--out /workspace/app/results/{SYM_INFER_NPZ_PATH.name}",
    ]
)

cp_symm_infer = run_docker(infer_cmd, check=False)
save_log(SYM_INFER_LOG, cp_symm_infer)

combined_infer = (cp_symm_infer.stdout or "") + "\n" + (cp_symm_infer.stderr or "")
metric_lines = [line for line in combined_infer.splitlines() if "RMSE=" in line]

print("=== SymMGN DOE 추론 (Docker) ===")
print(
    json.dumps(
        {
            "returncode": cp_symm_infer.returncode,
            "case_idx": DOE_INFER_CASE_IDX,
            "source_file_types": DOE_SOURCE_FILE_TYPES,
            "max_steps": DOE_INFER_MAX_STEPS,
            "infer_npz_exists": SYM_INFER_NPZ_PATH.exists(),
            "infer_npz_size_kb": round(SYM_INFER_NPZ_PATH.stat().st_size / 1024, 1) if SYM_INFER_NPZ_PATH.exists() else 0,
            "metric_lines": metric_lines[-4:],
            "log_file": str(LOG_DIR / SYM_INFER_LOG),
        },
        indent=2,
        ensure_ascii=False,
    )
)

if cp_symm_infer.returncode != 0:
    print("=== infer stderr tail ===")
    print("\n".join(combined_infer.splitlines()[-40:]))
    raise RuntimeError("DOE 실데이터 SymMGN 추론 실패. 12_symm_infer_doe.log를 확인하세요.")

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np

DOE_INFER_CASE_IDX = globals().get("DOE_INFER_CASE_IDX", 0)
SYM_INFER_NPZ_PATH = globals().get(
    "SYM_INFER_NPZ_PATH",
    ROOT / "results" / f"symm_mgn_case{DOE_INFER_CASE_IDX:04d}_onloadtorque_step1.npz",
)
SYM_VIS_PNG_PATH = ROOT / "logs" / f"symm_mgn_case{DOE_INFER_CASE_IDX:04d}_gt_vs_pred.png"

if not SYM_INFER_NPZ_PATH.exists():
    raise FileNotFoundError(f"추론 NPZ가 없습니다: {SYM_INFER_NPZ_PATH}")

arr = np.load(SYM_INFER_NPZ_PATH, allow_pickle=True)
pos_x = arr["pos_x"]
pos_y = arr["pos_y"]
channels = ["bx", "by", "a", "j"]
labels = ["Bx", "By", "A", "J"]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle(
    f"SymMGN DOE case {DOE_INFER_CASE_IDX:04d} - OnLoadTorque GT vs Prediction",
    fontsize=12,
)

for col_idx, (channel_name, label) in enumerate(zip(channels, labels)):
    gt_values = arr[f"gt_{channel_name}"]
    pred_values = arr[f"pred_{channel_name}"]
    vmin = float(min(gt_values.min(), pred_values.min()))
    vmax = float(max(gt_values.max(), pred_values.max()))

    gt_plot = axes[0, col_idx].scatter(
        pos_x,
        pos_y,
        c=gt_values,
        cmap="RdBu_r",
        s=10,
        vmin=vmin,
        vmax=vmax,
    )
    axes[0, col_idx].set_title(f"GT {label}")
    axes[0, col_idx].set_aspect("equal")
    axes[0, col_idx].set_xticks([])
    axes[0, col_idx].set_yticks([])
    plt.colorbar(gt_plot, ax=axes[0, col_idx], fraction=0.04)

    pred_plot = axes[1, col_idx].scatter(
        pos_x,
        pos_y,
        c=pred_values,
        cmap="RdBu_r",
        s=10,
        vmin=vmin,
        vmax=vmax,
    )
    axes[1, col_idx].set_title(f"Pred {label}")
    axes[1, col_idx].set_aspect("equal")
    axes[1, col_idx].set_xticks([])
    axes[1, col_idx].set_yticks([])
    plt.colorbar(pred_plot, ax=axes[1, col_idx], fraction=0.04)

plt.tight_layout()
SYM_VIS_PNG_PATH.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(SYM_VIS_PNG_PATH, dpi=110, bbox_inches="tight")
plt.show()
plt.close()

meta = json.loads(arr["meta"].item()) if "meta" in arr.files else {}
metrics = json.loads(arr["metrics"].item()) if "metrics" in arr.files else {}

print(
    json.dumps(
        {
            "saved_png": str(SYM_VIS_PNG_PATH),
            "png_size_kb": round(SYM_VIS_PNG_PATH.stat().st_size / 1024, 1),
            "meta": meta,
            "metrics": metrics,
        },
        indent=2,
        ensure_ascii=False,
    )
)

## 11) Full 40-case DOE 학습

DOE manifest의 전체 40개 케이스를 사용해 SymMGN을 본격 학습합니다.

| 항목 | 값 |
|------|----|
| 모델 | SymMGN (anti-periodic PBC) |
| Case | **40개** (DOE 전체) |
| Source file type | OnLoadTorque |
| Max steps/case | 1 (static 첫 step) |
| Epochs | 50 |
| Hidden dim | 128 |
| Batch size | 4 |

> ⚠ GPU 시간이 상당히 소요됩니다. 진행 상황은 로그 파일에서 확인하세요.

In [ ]:
import json

# ── Full 40-case training config ──
FULL_CASE_INDICES = list(range(40))  # DOE 전체 40 cases
FULL_SOURCE_FILE_TYPES = ["OnLoadTorque"]
FULL_MAX_STEPS_PER_CASE = 1  # static 첫 step만
FULL_EPOCHS = 50
FULL_BATCH_SIZE = 4
FULL_HIDDEN_DIM = 128
FULL_LR = 1e-3
FULL_SEED = 42

FULL_CKPT_PATH = ROOT / "results" / "symm_mgn_doe_full40_onloadtorque.pt"
FULL_TRAIN_LOG = "11_full40_train.log"

full_train_cmd = " ".join(
    [
        "python -m phase1_static.train",
        "--input-format doe",
        "--data-dir doe_data",
        "--case-indices " + " ".join(str(idx) for idx in FULL_CASE_INDICES),
        "--source-file-types " + " ".join(FULL_SOURCE_FILE_TYPES),
        f"--max-steps-per-case {FULL_MAX_STEPS_PER_CASE}",
        f"--epochs {FULL_EPOCHS}",
        f"--batch-size {FULL_BATCH_SIZE}",
        f"--hidden-dim {FULL_HIDDEN_DIM}",
        f"--lr {FULL_LR}",
        f"--seed {FULL_SEED}",
        f"--ckpt-out results/{FULL_CKPT_PATH.name}",
    ]
)

cp_full_train = run_docker(full_train_cmd, check=False)
save_log(FULL_TRAIN_LOG, cp_full_train)

combined_full = (cp_full_train.stdout or "") + "\n" + (cp_full_train.stderr or "")
epoch_lines = [l for l in combined_full.splitlines() if "epoch=" in l]

print("=== SymMGN Full 40-case 학습 (Docker) ===")
print(
    json.dumps(
        {
            "returncode": cp_full_train.returncode,
            "total_cases": len(FULL_CASE_INDICES),
            "epochs": FULL_EPOCHS,
            "hidden_dim": FULL_HIDDEN_DIM,
            "last_epoch": epoch_lines[-1] if epoch_lines else "",
            "ckpt_saved": FULL_CKPT_PATH.exists(),
            "ckpt_size_kb": round(FULL_CKPT_PATH.stat().st_size / 1024, 1) if FULL_CKPT_PATH.exists() else 0,
            "log_file": str(LOG_DIR / FULL_TRAIN_LOG),
        },
        indent=2,
        ensure_ascii=False,
    )
)

if cp_full_train.returncode != 0:
    print("=== train stderr tail ===")
    print("\n".join(combined_full.splitlines()[-40:]))
    raise RuntimeError("Full 40-case SymMGN 학습 실패. 11_full40_train.log를 확인하세요.")

## 12) Full 40-case 추론

학습된 Full 40-case 모델로 전체 케이스를 순회하며 추론합니다.
각 case별 per-channel RMSE를 수집해 통계를 생성합니다.

In [ ]:
import json
import numpy as np

# ── Full inference across all 40 cases ──
FULL_CKPT_PATH = globals().get("FULL_CKPT_PATH", ROOT / "results" / "symm_mgn_doe_full40_onloadtorque.pt")
FULL_INFER_DIR = ROOT / "results" / "full40_infer"
FULL_INFER_DIR.mkdir(parents=True, exist_ok=True)
FULL_INFER_LOG = "12_full40_infer.log"

all_metrics = {}
failed_cases = []

for case_idx in range(40):
    npz_name = f"full40_case{case_idx:04d}_step1.npz"
    infer_cmd = " ".join(
        [
            "python infer_phase1_pbc.py",
            f"--ckpt /workspace/app/results/{FULL_CKPT_PATH.name}",
            "--data-dir /workspace/app/doe_data",
            f"--case-idx {case_idx}",
            "--source-file-types OnLoadTorque",
            "--max-steps 1",
            "--batch-size 1",
            f"--out /workspace/app/results/full40_infer/{npz_name}",
        ]
    )
    cp = run_docker(infer_cmd, check=False)

    combined = (cp.stdout or "") + "\n" + (cp.stderr or "")
    metric_lines = [l for l in combined.splitlines() if "RMSE=" in l]

    if cp.returncode == 0:
        all_metrics[case_idx] = metric_lines[-4:] if len(metric_lines) >= 4 else metric_lines
    else:
        failed_cases.append(case_idx)

    if (case_idx + 1) % 10 == 0:
        print(f"  ... {case_idx + 1}/40 cases done")

save_log(FULL_INFER_LOG, cp)  # save last case log

print(f"\n=== Full 40-case Inference Summary ===")
print(f"Success: {len(all_metrics)}/40")
print(f"Failed: {failed_cases}")

# Parse RMSE values from metric lines
channel_rmses = {ch: [] for ch in ["Bx", "By", "A", "J"]}
for case_idx, lines in all_metrics.items():
    for line in lines:
        for ch in channel_rmses:
            if ch in line:
                try:
                    val = float(line.split("RMSE=")[1].split()[0])
                    channel_rmses[ch].append(val)
                except (IndexError, ValueError):
                    pass

print("\nPer-channel RMSE statistics (40 cases):")
for ch, vals in channel_rmses.items():
    if vals:
        arr = np.array(vals)
        print(f"  {ch}: mean={arr.mean():.6f}, std={arr.std():.6f}, max={arr.max():.6f}")

## 13) Full 40-case GT vs Pred 시각화

대표 case 4개를 골라 GT vs Pred scatter를 비교합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

FULL_INFER_DIR = globals().get("FULL_INFER_DIR", ROOT / "results" / "full40_infer")
VIS_CASE_INDICES = [0, 10, 20, 30]  # 대표 4개 case

channels = ["bx", "by", "a", "j"]
labels = ["Bx", "By", "A", "J"]

for case_idx in VIS_CASE_INDICES:
    npz_path = FULL_INFER_DIR / f"full40_case{case_idx:04d}_step1.npz"
    if not npz_path.exists():
        print(f"case {case_idx}: NPZ 없음, skip")
        continue

    arr = np.load(npz_path, allow_pickle=True)
    pos_x, pos_y = arr["pos_x"], arr["pos_y"]

    fig, axes = plt.subplots(2, 4, figsize=(18, 7))
    fig.suptitle(f"case {case_idx:04d} — GT (top) vs Pred (bottom)", fontsize=13)

    for col_idx, (ch, label) in enumerate(zip(channels, labels)):
        gt = arr[f"gt_{ch}"]
        pred = arr[f"pred_{ch}"]
        vmin = float(min(gt.min(), pred.min()))
        vmax = float(max(gt.max(), pred.max()))

        sc_gt = axes[0, col_idx].scatter(
            pos_x, pos_y, c=gt, cmap="RdBu_r", s=6, vmin=vmin, vmax=vmax,
        )
        axes[0, col_idx].set_title(f"GT {label}")
        axes[0, col_idx].set_aspect("equal")
        axes[0, col_idx].set_xticks([])
        axes[0, col_idx].set_yticks([])
        plt.colorbar(sc_gt, ax=axes[0, col_idx], fraction=0.04)

        sc_pred = axes[1, col_idx].scatter(
            pos_x, pos_y, c=pred, cmap="RdBu_r", s=6, vmin=vmin, vmax=vmax,
        )
        axes[1, col_idx].set_title(f"Pred {label}")
        axes[1, col_idx].set_aspect("equal")
        axes[1, col_idx].set_xticks([])
        axes[1, col_idx].set_yticks([])
        plt.colorbar(sc_pred, ax=axes[1, col_idx], fraction=0.04)

    plt.tight_layout()
    save_path = ROOT / "logs" / f"full40_case{case_idx:04d}_gt_vs_pred.png"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, dpi=110, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"saved: {save_path}")

## 13-B) Full 40-case |GT - Pred| Error Map

동일 대표 case 4개에 대해 채널별 absolute error를 시각화합니다.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

FULL_INFER_DIR = globals().get("FULL_INFER_DIR", ROOT / "results" / "full40_infer")
VIS_CASE_INDICES = [0, 10, 20, 30]  # 대표 4개 case

fig, axes = plt.subplots(len(VIS_CASE_INDICES), 4, figsize=(16, 4 * len(VIS_CASE_INDICES)))
fig.suptitle("Full 40-case SymMGN — GT vs Pred (대표 4 case)", fontsize=14)

channels = ["bx", "by", "a", "j"]
labels = ["Bx", "By", "A", "J"]

for row_idx, case_idx in enumerate(VIS_CASE_INDICES):
    npz_path = FULL_INFER_DIR / f"full40_case{case_idx:04d}_step1.npz"
    if not npz_path.exists():
        for col in range(4):
            axes[row_idx, col].text(0.5, 0.5, f"case {case_idx}\nNPZ 없음", ha="center", va="center")
        continue

    arr = np.load(npz_path, allow_pickle=True)
    pos_x, pos_y = arr["pos_x"], arr["pos_y"]

    for col_idx, (ch, label) in enumerate(zip(channels, labels)):
        gt = arr[f"gt_{ch}"]
        pred = arr[f"pred_{ch}"]
        error = np.abs(gt - pred)
        vmax_err = float(np.percentile(error, 95))

        sc = axes[row_idx, col_idx].scatter(
            pos_x, pos_y, c=error, cmap="hot_r", s=6, vmin=0, vmax=vmax_err,
        )
        axes[row_idx, col_idx].set_aspect("equal")
        axes[row_idx, col_idx].set_xticks([])
        axes[row_idx, col_idx].set_yticks([])
        if row_idx == 0:
            axes[row_idx, col_idx].set_title(f"|GT-Pred| {label}")
        if col_idx == 0:
            axes[row_idx, col_idx].set_ylabel(f"case {case_idx:04d}")
        plt.colorbar(sc, ax=axes[row_idx, col_idx], fraction=0.04)

plt.tight_layout()
vis_path = ROOT / "logs" / "full40_gt_vs_pred_error.png"
vis_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(vis_path, dpi=110, bbox_inches="tight")
plt.show()
plt.close()
print(f"saved: {vis_path}")

## 13-C) Anti-Periodic 대칭을 이용한 Full Motor 복원

1/8 섹터 추론 결과를 8회 회전 + anti-periodic sign flip으로 full 360° 모터 단면을 복원합니다.

**변환 규칙 (섹터 k = 0…7, α = 45°):**
- 위치: $(x, y)$ → 회전 $k\alpha$
- 스칼라 (A, J): $(-1)^k$ 부호 반전
- 벡터 (Bx, By): $(-1)^k$ × 회전 행렬 적용

$$\begin{bmatrix} B_x' \\ B_y' \end{bmatrix} = (-1)^k \begin{bmatrix} \cos k\alpha & -\sin k\alpha \\ \sin k\alpha & \cos k\alpha \end{bmatrix} \begin{bmatrix} B_x \\ B_y \end{bmatrix}$$

## 14) Phase 1 완료 Evidence Summary

Phase 1 (Static SymMGN with PBC) 완료 체크리스트:

| 항목 | 상태 |
|------|------|
| Contract 경계 테스트 통과 | ✅ 위 셀에서 확인 |
| PBC 경계/계약 호스트 테스트 | ✅ 위 셀에서 확인 |
| Overfit-Single 게이트 통과 | ✅ 위 셀에서 확인 |
| PBC 가시화 (학습 전) | ✅ 위 셀에서 확인 |
| 3-case Smoke Test 통과 | ✅ 위 셀에서 확인 |
| Full 40-case 학습 완료 | ⬜ 위 셀 실행 후 체크 |
| Full 40-case 추론 + RMSE 통계 | ⬜ 위 셀 실행 후 체크 |
| GT vs Pred 시각화 | ⬜ 위 셀 실행 후 체크 |

Phase 1이 완료되면 `phase2_tutorial.ipynb`로 진행합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

FULL_INFER_DIR = globals().get("FULL_INFER_DIR", ROOT / "results" / "full40_infer")
FULL_VIS_CASE = 0

SECTOR_COUNT = 8
SECTOR_ANGLE_DEG = 45.0


def reconstruct_full_motor(pos_x, pos_y, fields_dict, n_sectors=8, angle_deg=45.0):
    all_x, all_y = [], []
    all_fields = {key: [] for key in fields_dict}

    for sector_idx in range(n_sectors):
        rad = np.radians(sector_idx * angle_deg)
        cos_k = np.cos(rad)
        sin_k = np.sin(rad)
        sign = (-1.0) ** sector_idx

        rot_x = pos_x * cos_k - pos_y * sin_k
        rot_y = pos_x * sin_k + pos_y * cos_k
        all_x.append(rot_x)
        all_y.append(rot_y)

        bx_orig = fields_dict["bx"]
        by_orig = fields_dict["by"]
        all_fields["bx"].append(sign * (bx_orig * cos_k - by_orig * sin_k))
        all_fields["by"].append(sign * (bx_orig * sin_k + by_orig * cos_k))
        all_fields["a"].append(sign * fields_dict["a"])
        all_fields["j"].append(sign * fields_dict["j"])

    full_x = np.concatenate(all_x)
    full_y = np.concatenate(all_y)
    full_fields = {key: np.concatenate(values) for key, values in all_fields.items()}
    return full_x, full_y, full_fields


npz_path = FULL_INFER_DIR / f"full40_case{FULL_VIS_CASE:04d}_step1.npz"
if not npz_path.exists():
    raise FileNotFoundError(f"NPZ not found: {npz_path}")

arr = np.load(npz_path, allow_pickle=True)
sector_x = arr["pos_x"]
sector_y = arr["pos_y"]

gt_fields = {channel: arr[f"gt_{channel}"] for channel in ["bx", "by", "a", "j"]}
pred_fields = {channel: arr[f"pred_{channel}"] for channel in ["bx", "by", "a", "j"]}

gt_full_x, gt_full_y, gt_full = reconstruct_full_motor(sector_x, sector_y, gt_fields)
pred_full_x, pred_full_y, pred_full = reconstruct_full_motor(sector_x, sector_y, pred_fields)

channels = ["bx", "by", "a", "j"]
labels = ["Bx", "By", "A", "J"]

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
fig.suptitle(
    f"Full Motor (8 sectors) — case {FULL_VIS_CASE:04d} GT (top) vs Pred (bottom)",
    fontsize=14,
)

for col_idx, (channel, label) in enumerate(zip(channels, labels)):
    gt_vals = gt_full[channel]
    pred_vals = pred_full[channel]
    vmin = float(min(gt_vals.min(), pred_vals.min()))
    vmax = float(max(gt_vals.max(), pred_vals.max()))

    sc_gt = axes[0, col_idx].scatter(
        gt_full_x, gt_full_y, c=gt_vals, cmap="RdBu_r", s=1.5, vmin=vmin, vmax=vmax,
    )
    axes[0, col_idx].set_title(f"GT {label}")
    axes[0, col_idx].set_aspect("equal")
    axes[0, col_idx].set_xticks([])
    axes[0, col_idx].set_yticks([])
    plt.colorbar(sc_gt, ax=axes[0, col_idx], fraction=0.046)

    sc_pred = axes[1, col_idx].scatter(
        pred_full_x, pred_full_y, c=pred_vals, cmap="RdBu_r", s=1.5, vmin=vmin, vmax=vmax,
    )
    axes[1, col_idx].set_title(f"Pred {label}")
    axes[1, col_idx].set_aspect("equal")
    axes[1, col_idx].set_xticks([])
    axes[1, col_idx].set_yticks([])
    plt.colorbar(sc_pred, ax=axes[1, col_idx], fraction=0.046)

plt.tight_layout()
save_path = ROOT / "logs" / f"full_motor_case{FULL_VIS_CASE:04d}_gt_vs_pred.png"
save_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(save_path, dpi=130, bbox_inches="tight")
plt.show()
plt.close()
print(f"saved: {save_path}")
print(f"total nodes: {len(gt_full_x)} ({len(sector_x)} x {SECTOR_COUNT} sectors)")